# ResNet-18 on full CIFAR-10: raw vs invariant sharpness

This notebook runs the ResNet-18 experiment of *Flat in the Wrong Coordinates* on a free GPU and saves every result.
**You only need to start it; everything else is automatic.**

### Kaggle (recommended: 2 GPUs, about 6 hours, runs in the background)
1. kaggle.com → **Create → New Notebook** → **File → Import Notebook** → upload this `.ipynb`.
2. Right-hand panel, **Settings**: **Accelerator → GPU T4 x2**, and **Internet → On** (needs a phone-verified account).
3. Top right **Save Version** → **Save & Run All (Commit)** → **Save**. You can close the browser; it runs for up to 12 h.
4. When the version shows *Complete*: open it → **Output** → download `resnet_results.zip`.

If it stopped before finishing (the log says `TIME BUDGET REACHED`): open the notebook, **Add Input** → *Notebook Output Files* → pick this notebook's previous version, then Save & Run All again. It resumes where it stopped.

### Google Colab (1 GPU, about 11 hours over several sessions)
1. colab.research.google.com → **File → Upload notebook** → this `.ipynb`.
2. **Runtime → Change runtime type → T4 GPU**.
3. **Runtime → Run all**, and allow access to Google Drive (results are saved there, so nothing is lost if Colab disconnects).
4. If Colab disconnects, reconnect and **Run all** again: finished runs are skipped and interrupted runs resume.
5. At the end, download `resnet_results.zip` (it is also in Drive, folder `LossLandscape_resnet`).

### Sending the results back
Upload the two folders inside the zip (`resnet18_noaug/`, `resnet18_aug/`) to GitHub at
`LossLandscapeProject/research/results/` (**Add file → Upload files**), or attach the zip in the chat.


In [ ]:
# 1. Where are we, and which GPU do we have?
import os, sys, subprocess, shutil, glob, time
ON_KAGGLE = os.path.exists('/kaggle/working')
ON_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
print('Kaggle' if ON_KAGGLE else 'Colab' if ON_COLAB else 'local machine')
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout or 'no GPU found')
import torch
N_GPU = torch.cuda.device_count()
assert N_GPU > 0 or os.environ.get('ALLOW_CPU'), 'No GPU: on Kaggle set Accelerator = GPU T4 x2; on Colab Runtime -> Change runtime type -> T4 GPU'
print('GPUs:', N_GPU, '| torch', torch.__version__)

In [ ]:
# 2. Results folder (Colab: Google Drive, so that a disconnect loses nothing)
if ON_KAGGLE:
    OUT = '/kaggle/working/resnet_results'
    WORK = '/kaggle/working'
    TIME_BUDGET_H = 11.0          # a Kaggle commit run may last 12 h; stop new work after 11 h
elif ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/LossLandscape_resnet/resnet_results'
    WORK = '/content'
    TIME_BUDGET_H = 100.0         # Colab: just run; rerun the notebook after a disconnect
else:
    WORK = os.environ.get('WORK', os.getcwd())
    OUT = os.environ.get('OUT', os.path.join(WORK, 'resnet_results'))
    TIME_BUDGET_H = float(os.environ.get('TIME_BUDGET_H', 100))
os.makedirs(OUT, exist_ok=True)
# Kaggle: continue from a previous version's output if it was added as an input
if ON_KAGGLE:
    for prev in glob.glob('/kaggle/input/**/resnet_results', recursive=True):
        for f in glob.glob(os.path.join(prev, '**', '*'), recursive=True):
            if os.path.isfile(f):
                dst = os.path.join(OUT, os.path.relpath(f, prev))
                if not os.path.exists(dst):
                    os.makedirs(os.path.dirname(dst), exist_ok=True)
                    shutil.copy2(f, dst)
        print('resumed from', prev)
print('results go to', OUT)

In [ ]:
# 3. Get the code (public repository)
REPO = os.environ.get('REPO_DIR') or os.path.join(WORK, 'LossLandscape-Research')
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '--depth', '1', '--no-single-branch',
                    'https://github.com/Ashvinj2004/LossLandscape-Research.git', REPO], check=True)
    # use the working branch if it exists (it carries the newest code), otherwise main
    r = subprocess.run(['git', '-C', REPO, 'checkout', 'main-83mrs1'], capture_output=True, text=True)
    print('branch:', 'main-83mrs1' if r.returncode == 0 else 'main')
SCRIPT = os.path.join(REPO, 'LossLandscapeProject', 'research', 'scripts', 'resnet_cifar10.py')
DATA = os.environ.get('DATA_DIR') or os.path.join(WORK, 'cifar_data')
print(subprocess.run(['git', '-C', REPO, 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout)

In [ ]:
# 4. Self-test on this GPU (about 2 minutes): checks the maths, downloads CIFAR-10, estimates the run time
r = subprocess.run([sys.executable, SCRIPT, '--selftest', '--time_estimate', '--data', DATA] + (['--device', 'cpu', '--width_test', '4'] if N_GPU == 0 else []),
                   capture_output=True, text=True)
print(r.stdout[-4000:], r.stderr[-3000:])
assert 'SELFTEST PASSED' in r.stdout, 'Self-test failed: please send this output back'

In [ ]:
# 5. The experiment. Two sweeps: the paper's protocol (no augmentation, constant learning rate) first,
#    then a standard recipe (augmentation, weight decay, cosine schedule). With two GPUs each gets half of the runs.
SWEEPS = os.environ.get('SWEEPS', 'resnet18_noaug,resnet18_aug').split(',')
EXTRA = os.environ.get('EXTRA_ARGS', '').split()
t_start = time.time()
for sweep in SWEEPS:
    n = max(N_GPU, 1)
    procs = []
    for k in range(n):
        env = dict(os.environ)
        if N_GPU > 0:
            env['CUDA_VISIBLE_DEVICES'] = str(k)
        budget = max(0.1, TIME_BUDGET_H - (time.time() - t_start) / 3600)
        log = open(os.path.join(OUT, f'{sweep}_shard{k}.log'), 'a')
        procs.append((subprocess.Popen([sys.executable, '-u', SCRIPT, '--sweep', sweep, '--out', OUT, '--data', DATA,
                                        '--shard', f'{k}/{n}', '--time_budget_h', f'{budget:.2f}'] + EXTRA,
                                       stdout=log, stderr=subprocess.STDOUT, env=env), log))
    # live progress: last line of every shard's log, every 2 minutes
    while any(p.poll() is None for p, _ in procs):
        time.sleep(120 if N_GPU else 10)
        lines = []
        for k in range(n):
            with open(os.path.join(OUT, f'{sweep}_shard{k}.log')) as f:
                tail = [l.strip() for l in f.readlines() if l.strip()]
            lines.append(f'  shard {k}: ' + (tail[-1] if tail else '(starting)'))
        print(f'[{(time.time() - t_start) / 3600:.2f} h] {sweep}', *lines, sep='\n', flush=True)
    for p, log in procs:
        log.close()
    done = len(glob.glob(os.path.join(OUT, sweep, '*.json'))) - len(glob.glob(os.path.join(OUT, sweep, '*.partial.json')))
    print(f'{sweep}: {done} runs finished', flush=True)
    if any('TIME BUDGET REACHED' in open(os.path.join(OUT, f'{sweep}_shard{k}.log')).read() for k in range(n)):
        print('Time budget reached: run the notebook again to continue (see the instructions at the top).')
        break

In [ ]:
# 6. Summary and a zip of everything to send back
import json
for sweep in SWEEPS:
    files = sorted(f for f in glob.glob(os.path.join(OUT, sweep, '*.json')) if not f.endswith('.partial.json'))
    print(f'\n{sweep}: {len(files)} finished runs')
    for f in files:
        r = json.load(open(f))
        last = r['ckpts'][-1] if r['ckpts'] else {}
        print(f"  {os.path.basename(f)[:-5]:42s} {r['status']:10s} epochs {r['epochs']:3d} "
              f"test acc {last.get('test_acc', float('nan')):.3f} checkpoints {[c['target'] for c in r['ckpts']]}")
import zipfile
zip_base = os.path.join(WORK, 'resnet_results')
with zipfile.ZipFile(zip_base + '.zip', 'w', zipfile.ZIP_DEFLATED) as z:   # results and logs only
    for f in glob.glob(os.path.join(OUT, '**', '*'), recursive=True):
        if f.endswith(('.json', '.log')):
            z.write(f, os.path.relpath(f, os.path.dirname(OUT)))
print('\nzip:', zip_base + '.zip', f'({os.path.getsize(zip_base + ".zip") / 1e6:.1f} MB)')
if ON_COLAB:
    from google.colab import files
    files.download(zip_base + '.zip')